In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
try:
    import lm_eval
except ImportError:
    %pip install -q git+https://github.com/EleutherAI/lm-evaluation-harness
    import lm_eval

# Constants

In [3]:
TAWJEEH_DATASET_NAME = 'belebele'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/belebele_experimental'
TASK_NAME='NLU'
MODEL_PATH = "/hdd/shared_models/AceGPT-7B"
TOKENIZER_PATH = MODEL_PATH
BATCH_SIZE = 40
# ------------------------
MODEL_NAME = MODEL_PATH.split('/')[-1]
# -----------------------
TUNED_MODEL_PATH = f"Notebooks/Experiments/cross_tasks_tuning/tuned_models/{MODEL_NAME}"

# Building the prompts dataset

In [ ]:
import requests
 
from tqdm.auto import tqdm
 
prompts = None
 
tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://tawjeeh.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts:
    raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14892,
  'tags': [],
  'name': 'mais-prompt1',
  'task': {'name': 'NLI'},
  'status': 'SUBMITTED',
  'template': 'Premise: {{ Premise }}\r\nHypothesis: {{ Hypothesis }}\r\n\r\nDoes the hypothesis about the premise entails? (Yes or No)\r\nAnswer:\r\n|||\r\n{{ answer_choices[label] }}',
  'created_by': 'mais',
  'dataset_name': 'arbml/ArabicTE',
  'dataset_subset': 'default',
  'answer_choices': ['No', 'Yes'],
  'text_direction': 'ltr'},
 {'id': 14891,
  'tags': [],
  'name': 'expert Arabic summarizer',
  'task': {'name': 'summarization'},
  'status': 'APPROVED',
  'template': 'You are an expert Arabic text summarizer. The following article:\r\n{{article}}\xa0\r\ncan be summarized as:\r\n|||\r\n{{summary}}',
  'created_by': 'majed.alshaibani',
  'dataset_name': 'arbml/AraSum',
  'dataset_subset': 'default',
  'answer_choices': [],
  'text_direction': 'ltr'},
 {'id': 14890,
  'tags': [],
  'name': 'Translation as completion',
  'task': {'name': 'machine translation'},
  'status': 

filter prompts:
- get only the approved ones
- get only the ones on the sarcasim detection datasets (emotone_ar,sem_eval_2018_task_1)

In [5]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

235

### Get the dataset prompts

In [6]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

5

In [7]:
SELECTED_PROMPTS_IDS = [
    14854,
    14853,   
    14801,
    14800,
    14575,
]

In [8]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the experimental dataset

In [9]:
import datasets

In [10]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    test: Dataset({
        features: ['link', 'question_number', 'flores_passage', 'question', 'mc_answer1', 'mc_answer2', 'mc_answer3', 'mc_answer4', 'correct_answer_num', 'dialect', 'ds'],
        num_rows: 900
    })
})

### Merge the prompts

In [11]:
from jinja2 import Environment, StrictUndefined

In [12]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        sample['answer_choices'] = prompt_template['answer_choices']
        env = Environment(undefined=StrictUndefined)
        if "|||" not in template:
            raise ValueError("No ||| dividor")
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e
        

Perform generation on one example prompt, for experimentation

In [13]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['test'][1]))

Given the following passage, query, and answer choices, output the letter corresponding to the correct answer.
###
Passage: """جزر كوك هي دولة جزرية مرتبطة بشكل حر بنيوزيلندا، وتصير ببولينيزيا وسط جنوب المحيط الهادئ. هي أرخبيل بي 15 جزيرة منتشرة على مساحة تزيد عن 2.2 مليون كيلومتر مربع من المحيط. تتشارك بنفس المنطقة الزمنية مثل هاواي، وتعتبر أحياناً هذي الجزر بأنها """"هاواي أسفل"""". على الرغم من كونها أصغر بالحجم، بس تذكر بعض الزائرين القدامى اللي جايين من هاواي ما قبل قيام الدولة قبل وجود كل الفنادق السياحية الكبيرة وغيرها من التطويرات. ماكو بجزر كوك أي مدن بس تتكون من 15 جزيرة أهمها راروتونجا وإيتوتاكي."""
###
Query: أي من التالي ما يوصف جزر كوك بشكل دقيق؟ 
###
Choices:


A. اصغر من هاواي

B. عبارة عن أرخبيل 

C. مدنها الرئيسية هي راروتونجا وإيتوتاكي

D. البلد الجزري يتشارك نفس المنطقة الزمنيه ويه هاواي

###
Answer:
|||
C


In [14]:
for prompt in dataset_prompts:
    prompt['merged_samples'] = list(
        map(
            lambda sample: apply_template(prompt, sample),
            # hf_exp_dataset['test'].select(range(100)),
            tqdm(hf_exp_dataset['test']),
        )
    )
    prompt['original_samples'] = list(hf_exp_dataset['test'])

  0%|          | 0/900 [00:00<?, ?it/s]

  0%|          | 0/900 [00:00<?, ?it/s]

  0%|          | 0/900 [00:00<?, ?it/s]

  0%|          | 0/900 [00:00<?, ?it/s]

  0%|          | 0/900 [00:00<?, ?it/s]

# Evaluate on each prompt and report the results

In [15]:
from datasets import DatasetDict
import re

def create_hf_dataset(dataset_prompt, columns=None):
  if columns is None:
    columns = ['text', 'label','choices']
  texts = []
  labels = []
  choices = []
  for i,merged_sample in enumerate(dataset_prompt['merged_samples']):
    # prefix= merged_sample.split('|||')[0].replace('\n', ' ')
    prefix= merged_sample.split('|||')[0]
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    prefix = prefix.strip()
    # add new line after prefx (this turns out to have a large effect on some prompts)
    # prefix = prefix+'\n'
    output = merged_sample.split('|||')[1].replace('\n', '').strip()
    example_choices = dataset_prompt['answer_choices']
    texts.append(prefix)
    labels.append(output)
    choices.append(example_choices)
  dataset = DatasetDict({ 'test' : datasets.Dataset.from_dict({
      columns[0]: texts,
      columns[1]: labels,
      columns[2]: choices,
  })})
  return dataset

In [16]:
dataset = create_hf_dataset(dataset_prompts[0])
dataset['test'][1]['text']

'Given the following passage, query, and answer choices, output the letter corresponding to the correct answer.\n###\nPassage: """جزر كوك هي دولة جزرية مرتبطة بشكل حر بنيوزيلندا، وتصير ببولينيزيا وسط جنوب المحيط الهادئ. هي أرخبيل بي 15 جزيرة منتشرة على مساحة تزيد عن 2.2 مليون كيلومتر مربع من المحيط. تتشارك بنفس المنطقة الزمنية مثل هاواي، وتعتبر أحياناً هذي الجزر بأنها """"هاواي أسفل"""". على الرغم من كونها أصغر بالحجم، بس تذكر بعض الزائرين القدامى اللي جايين من هاواي ما قبل قيام الدولة قبل وجود كل الفنادق السياحية الكبيرة وغيرها من التطويرات. ماكو بجزر كوك أي مدن بس تتكون من 15 جزيرة أهمها راروتونجا وإيتوتاكي."""\n###\nQuery: أي من التالي ما يوصف جزر كوك بشكل دقيق؟\xa0\n###\nChoices:\n\n\nA. اصغر من هاواي\n\nB. عبارة عن أرخبيل \n\nC. مدنها الرئيسية هي راروتونجا وإيتوتاكي\n\nD. البلد الجزري يتشارك نفس المنطقة الزمنيه ويه هاواي\n\n###\nAnswer:'

In [17]:
from lm_eval.models.huggingface import HFLM
kwargs = dict(
    pretrained=MODEL_PATH,
    trust_remote_code=True,
    parallelize=True,
    device_map="auto",
    tokenizer=TOKENIZER_PATH,
    batch_size=BATCH_SIZE,
)

if TUNED_MODEL_PATH:
    kwargs['peft'] = TUNED_MODEL_PATH

if 'lm_obj' not in locals():
    lm_obj = HFLM(**kwargs)

2024-12-09:17:58:44,557 INFO     [huggingface.py:483] Using model type 'default'
2024-12-09:17:58:45,024 INFO     [huggingface.py:350] Model parallel was set to True, setting max memory per GPU to {0: 84523417600, 1: 84523417600} and device map to 'auto'
/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is 

In [18]:
def evaluate_tasks(tasks,dataset_sub_path=TAWJEEH_DATASET_NAME):
    # MAKE SURE THE NOTEBOOK IS RUNNING FROM THE PROJECT ROOT!
    task_manager = lm_eval.tasks.TaskManager(include_path=f"eval_harness_extra_tasks/{dataset_sub_path}")
    results = lm_eval.simple_evaluate(  # call simple_evaluate
        model=lm_obj,
        tasks=tasks,
        num_fewshot=0,
        task_manager=task_manager,
    )
    return results

In [19]:
import json

def create_and_evaluate_single_prompt(prompt, save_results=True, force_re_evaluate=False):
    prompt_id = prompt['id']
    results_dir = f'evaluation_results/{MODEL_NAME}-cross-tasks-tuned/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    prompt_results_file_path = f'{results_dir}/prompt_{prompt_id}.json'
    
    # Check if results exist and handle based on parameters
    if os.path.exists(prompt_results_file_path) and os.path.getsize(prompt_results_file_path) > 0:
        if not force_re_evaluate:
            print(f"Skipping prompt {prompt_id} - results already exist")
            with open(prompt_results_file_path, 'r') as f:
                prompt_results = json.load(f)
                print(lm_eval.utils.make_table(prompt_results))
                return prompt_results
        else:
            print(f"Force re-evaluate enabled - reevaluating prompt {prompt_id}")
    
    # Create dataset and task files
    dataset = create_hf_dataset(prompt)
    
    # Save dataset
    dataset_dir = f'experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}'
    os.makedirs(dataset_dir, exist_ok=True)
    dataset['test'].to_parquet(f"{dataset_dir}/data.parquet")
    
    # Create YAML configuration
    yaml_text = f'''task: {TAWJEEH_DATASET_NAME}_prompt_{prompt_id}
dataset_path: experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}
output_type: multiple_choice
test_split: train
doc_to_text: text
doc_to_choice: choices
doc_to_target: label
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: True
  - metric: acc_norm
    aggregation: mean
    higher_is_better: true
metadata:
  version: 1.0'''
    
    # Save YAML
    yaml_dir = f'eval_harness_extra_tasks/{TAWJEEH_DATASET_NAME}'
    os.makedirs(yaml_dir, exist_ok=True)
    with open(f'{yaml_dir}/prompt_{prompt_id}.yaml', 'w') as f:
        f.write(yaml_text)
    
    # Evaluate single prompt
    evaluation_task_name = f'{TAWJEEH_DATASET_NAME}_prompt_{prompt_id}'
    prompt_results = evaluate_tasks(tasks=[evaluation_task_name])
    
    print(lm_eval.utils.make_table(prompt_results))
    
    # Save results if save_results is True
    if save_results:
        os.makedirs(results_dir, exist_ok=True)
        with open(prompt_results_file_path, 'w') as f:
            json.dump(prompt_results, f, ensure_ascii=False, indent=4, 
                     default=lambda o: '<not serializable>')
        print(f"Saved results for prompt {prompt_id}")
    else:
        print(f"Results not saved for prompt {prompt_id} (save_results=False)")
    
    print(f"Completed evaluation for prompt {prompt_id}")
    return prompt_results

In [20]:
def evaluate_all_prompts_sequentially(dataset_prompts, **kwargs):
    print(f"Starting sequential evaluation of {len(dataset_prompts)} prompts")
    all_results = {}
    
    for i, prompt in enumerate(dataset_prompts, 1):
        print('-' * 80)
        print(f"\nProcessing prompt {i}/{len(dataset_prompts)} (ID: {prompt['id']})")
        print("Template:", prompt['template'])
        print('-' * 80)
        
        prompt_results = create_and_evaluate_single_prompt(prompt, **kwargs)
        all_results[f"{TAWJEEH_DATASET_NAME}_prompt_{prompt['id']}"] = prompt_results
    
    return {'results': all_results}

In [21]:
all_results = evaluate_all_prompts_sequentially(dataset_prompts=dataset_prompts,force_re_evaluate=True)

Starting sequential evaluation of 5 prompts
--------------------------------------------------------------------------------

Processing prompt 1/5 (ID: 14854)
Template: Given the following passage, query, and answer choices, output the letter corresponding to the correct answer.
###
Passage: {{flores_passage}}
###
Query: {{question}} 
###
Choices:
{% set choices = [mc_answer1, mc_answer2, mc_answer3, mc_answer4] %}
{% for choice in choices %}
{{ answer_choices[loop.index0] }}. {{choice}}
{% endfor %}
###
Answer:
|||
{{ answer_choices[correct_answer_num|int-1] }}
--------------------------------------------------------------------------------
Force re-evaluate enabled - reevaluating prompt 14854


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

2024-12-09:17:59:14,230 INFO     [evaluator.py:164] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2024-12-09:17:59:14,235 INFO     [evaluator.py:217] Using pre-initialized model


Generating train split: 0 examples [00:00, ? examples/s]

2024-12-09:17:59:14,304 WARNING  [task.py:325] [Task: belebele_prompt_14854] has_training_docs and has_validation_docs are False, using test_docs as fewshot_docs but this is not recommended.
2024-12-09:17:59:14,305 WARNING  [task.py:325] [Task: belebele_prompt_14854] has_training_docs and has_validation_docs are False, using test_docs as fewshot_docs but this is not recommended.
2024-12-09:17:59:14,334 WARNING  [evaluator.py:270] Overwriting default num_fewshot of belebele_prompt_14854 from None to 0
2024-12-09:17:59:14,335 INFO     [task.py:415] Building contexts for belebele_prompt_14854 on rank 0...
100%|██████████| 900/900 [00:00<00:00, 89566.59it/s]
2024-12-09:17:59:14,378 INFO     [evaluator.py:489] Running loglikelihood requests
Running loglikelihood requests: 100%|██████████| 3600/3600 [00:50<00:00, 71.35it/s] 
2024-12-09:18:00:07,571 WARNING  [huggingface.py:1375] Failed to get model SHA for /hdd/shared_models/AceGPT-7B at revision main. Error: Repo id must be in the form 'rep

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

2024-12-09:18:03:09,707 INFO     [evaluator.py:164] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2024-12-09:18:03:09,709 INFO     [evaluator.py:217] Using pre-initialized model


Generating train split: 0 examples [00:00, ? examples/s]

2024-12-09:18:03:09,769 WARNING  [task.py:325] [Task: belebele_prompt_14575] has_training_docs and has_validation_docs are False, using test_docs as fewshot_docs but this is not recommended.
2024-12-09:18:03:09,770 WARNING  [task.py:325] [Task: belebele_prompt_14575] has_training_docs and has_validation_docs are False, using test_docs as fewshot_docs but this is not recommended.
2024-12-09:18:03:09,797 WARNING  [evaluator.py:270] Overwriting default num_fewshot of belebele_prompt_14575 from None to 0
2024-12-09:18:03:09,799 INFO     [task.py:415] Building contexts for belebele_prompt_14575 on rank 0...
100%|██████████| 900/900 [00:00<00:00, 100948.64it/s]
2024-12-09:18:03:09,837 INFO     [evaluator.py:489] Running loglikelihood requests
Running loglikelihood requests: 100%|██████████| 3600/3600 [00:42<00:00, 84.77it/s] 
2024-12-09:18:03:55,279 WARNING  [huggingface.py:1375] Failed to get model SHA for /hdd/shared_models/AceGPT-7B at revision main. Error: Repo id must be in the form 're

|        Tasks        |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|---------------------|------:|------|-----:|--------|---|-----:|---|-----:|
|belebele_prompt_14575|      1|none  |     0|acc     |↑  |0.2378|±  |0.0142|
|                     |       |none  |     0|acc_norm|↑  |0.2378|±  |0.0142|

Saved results for prompt 14575
Completed evaluation for prompt 14575


In [ ]:
exit()

: 